In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("../data/accepted_2007_to_2018Q4.csv.gz")

/var/folders/qk/4jkhsyk50473tc0y7b0p8pww0000gn/T/ipykernel_12775/2493494034.py:1: DtypeWarning: Columns (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/accepted_2007_to_2018Q4.csv.gz")


In [4]:
df.shape

(2260701, 151)

In [5]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 2.5+ GB


In [7]:
df.columns.tolist()

['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 '

In [8]:
df["loan_status"].value_counts(dropna=False)

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64

In [9]:
analysis_df = df[
    df["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])
].copy()

analysis_df["default_flag"] = analysis_df["loan_status"].isin(
    ["Charged Off", "Default"]
).astype(int)

In [10]:
analysis_df["default_flag"].value_counts()

default_flag
0    1076751
1     268599
Name: count, dtype: int64

In [11]:
analysis_df["default_flag"].value_counts(normalize=True)

default_flag
0    0.80035
1    0.19965
Name: proportion, dtype: float64

In [12]:
important_cols = [
    "loan_amnt",
    "int_rate",
    "annual_inc",
    "dti",
    "fico_range_low",
    "fico_range_high",
    "emp_length",
    "purpose",
    "term",
    "loan_status"
]

analysis_df[important_cols].head()

,loan_amnt,int_rate,annual_inc,dti,fico_range_low,fico_range_high,emp_length,purpose,term,loan_status
0,3600.0,13.99,55000.0,5.91,675.0,679.0,10+ years,debt_consolidation,36 months,Fully Paid
1,24700.0,11.99,65000.0,16.06,715.0,719.0,10+ years,small_business,36 months,Fully Paid
2,20000.0,10.78,63000.0,10.78,695.0,699.0,10+ years,home_improvement,60 months,Fully Paid
4,10400.0,22.45,104433.0,25.37,695.0,699.0,3 years,major_purchase,60 months,Fully Paid
5,11950.0,13.44,34000.0,10.20,690.0,694.0,4 years,debt_consolidation,36 months,Fully Paid


In [13]:
analysis_df[important_cols].isnull().sum()

loan_amnt              0
int_rate               0
annual_inc             0
dti                  374
fico_range_low         0
fico_range_high        0
emp_length         78516
purpose                0
term                   0
loan_status            0
dtype: int64

In [15]:
missing_summary = pd.DataFrame({
    "missing_count": analysis_df[important_cols].isnull().sum(),
    "missing_percent": analysis_df[important_cols].isnull().mean() * 100
})

missing_summary.round(2)

,missing_count,missing_percent
loan_amnt,0,0.00
int_rate,0,0.00
annual_inc,0,0.00
dti,374,0.03
fico_range_low,0,0.00
fico_range_high,0,0.00
emp_length,78516,5.84
purpose,0,0.00
term,0,0.00
loan_status,0,0.00


In [16]:
analysis_df.duplicated().sum()

np.int64(0)

In [17]:
numerical_vars = [
    "loan_amnt",
    "int_rate",
    "annual_inc",
    "dti",
    "fico_range_low",
    "fico_range_high"
]

categorical_vars = [
    "emp_length",
    "purpose",
    "term",
    "loan_status"
]

target_var = "default_flag"

print("Numerical:", numerical_vars)
print("Categorical:", categorical_vars)
print("Target:", target_var)

Numerical: ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'fico_range_high']
Categorical: ['emp_length', 'purpose', 'term', 'loan_status']
Target: default_flag


In [18]:
analysis_df.groupby("loan_status")["default_flag"].agg(["count", "mean"])

,count,mean
loan_status,,
Charged Off,268559,1.0
Default,40,1.0
Fully Paid,1076751,0.0


In [19]:
data_dictionary = pd.DataFrame({
    "variable": [
        "loan_amnt",
        "int_rate",
        "annual_inc",
        "dti",
        "fico_range_low",
        "fico_range_high",
        "emp_length",
        "purpose",
        "term",
        "loan_status",
        "default_flag"
    ],
    
    "description": [
        "Original loan amount",
        "Loan interest rate",
        "Borrower annual income",
        "Debt-to-income ratio",
        "Lower bound of borrower FICO score",
        "Upper bound of borrower FICO score",
        "Borrower employment length",
        "Purpose of the loan",
        "Loan repayment term",
        "Final loan status",
        "Default indicator: 1 = default, 0 = non-default"
    ],
    
    "type": [
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Numeric",
        "Categorical",
        "Categorical",
        "Categorical",
        "Categorical",
        "Binary"
    ]
})

data_dictionary

,variable,description,type
0,loan_amnt,Original loan amount,Numeric
1,int_rate,Loan interest rate,Numeric
2,annual_inc,Borrower annual income,Numeric
3,dti,Debt-to-income ratio,Numeric
4,fico_range_low,Lower bound of borrower FICO score,Numeric
5,fico_range_high,Upper bound of borrower FICO score,Numeric
6,emp_length,Borrower employment length,Categorical
7,purpose,Purpose of the loan,Categorical
8,term,Loan repayment term,Categorical
9,loan_status,Final loan status,Categorical


In [20]:
portfolio_stats = pd.Series({
    "Total Loans": len(analysis_df),
    "Total Loan Amount": analysis_df["loan_amnt"].sum(),
    "Average Loan Amount": analysis_df["loan_amnt"].mean(),
    "Average Interest Rate": analysis_df["int_rate"].mean(),
    "Average Annual Income": analysis_df["annual_inc"].mean(),
    "Average DTI": analysis_df["dti"].mean(),
    "Defaulted Loans": analysis_df["default_flag"].sum(),
    "Default Rate (%)": analysis_df["default_flag"].mean() * 100
})

portfolio_stats.round(2)

Total Loans              1.345350e+06
Total Loan Amount        1.939991e+10
Average Loan Amount      1.441997e+04
Average Interest Rate    1.324000e+01
Average Annual Income    7.624757e+04
Average DTI              1.828000e+01
Defaulted Loans          2.685990e+05
Default Rate (%)         1.996000e+01
dtype: float64